In [44]:
# ============================================================
# IF_v2 공공데이터 전체 수집 + 파일 저장
# 11개 API 서비스 / 12개 수집 작업
# XML + odcloud JSON
# ============================================================

import os
import re
import json
import time
import hashlib
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from urllib.parse import unquote
from dotenv import load_dotenv
from IPython.display import display


# ============================================================
# 0. 프로젝트 경로 / .env 자동 탐색
# ============================================================

cwd = Path.cwd().resolve()

candidates = [
    cwd,
    cwd / "data",
    *cwd.parents,
]

project_root = next(
    (
        p for p in candidates
        if (p / "data").is_dir()
    ),
    cwd
)

data_root = project_root / "data"

if not data_root.is_dir():
    data_root = cwd / "data"

data_root.mkdir(parents=True, exist_ok=True)

env_candidates = [
    data_root / ".env",
    project_root / ".env",
    cwd / ".env",
    cwd / "data" / ".env",
]

env_path = next(
    (p for p in env_candidates if p.is_file()),
    None
)

if env_path:
    load_dotenv(env_path, override=True)
    print("환경설정 파일 확인 완료")
else:
    print("안내: .env 파일이 없어 환경변수를 확인합니다.")

KEY = (
    os.getenv("DATA_GO_KR_API_KEY")
    or os.getenv("PUBLIC_DATA_API_KEY")
    or os.getenv("SERVICE_KEY")
)

if not KEY:
    raise RuntimeError(
        "API 인증키가 없습니다. "
        "data/.env 파일에 DATA_GO_KR_API_KEY를 설정하세요."
    )

KEY = unquote(KEY).strip()

# ============================================================
# 1. 저장 폴더
# ============================================================

BASE_DIR = data_root / "output" / "public_api"

RAW_DIR = BASE_DIR / "raw"
MERGED_DIR = BASE_DIR / "merged"
REPORT_DIR = BASE_DIR / "reports"

for folder in [RAW_DIR, MERGED_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("저장 경로:", BASE_DIR)


# ============================================================
# 2. 실행 설정
# ============================================================

CONNECT_TIMEOUT = 5
READ_TIMEOUT = 10

PAGE_SIZE = 1000
MAX_PAGES = 30
RETRIES = 2

# False: 이미 저장된 파일 재사용
# True: 모든 API 다시 수집
FORCE_REFRESH = False

# False: 전체 데이터 수집
# True: 첫 페이지 확인만
TEST_MODE = False

if TEST_MODE:
    MAX_PAGES = 1

XML_BASE = "https://apis.data.go.kr/B552474"
OD_BASE = "https://api.odcloud.kr/api"


# ============================================================
# 3. XML API 목록
# ============================================================

XML_SOURCES = {

    "PUB-001_자립형일자리_행정구역코드": {
        "base": f"{XML_BASE}/JobBsnInfoService",
        "operation": "getJobAdministCodeList",
    },

    "PUB-002A_노인사회활동_지역코드": {
        "base": f"{XML_BASE}/OdsnCodeInquiryService2",
        "operation": "getOdsnAreaCodeInquiryList2",
    },

    "PUB-002B_노인사회활동_기관코드": {
        "base": f"{XML_BASE}/OdsnCodeInquiryService2",
        "operation": "getOdsnOrgCodeInquiryList2",
    },

    "PUB-003_자립형일자리_수행기관": {
        "base": f"{XML_BASE}/JobBsnInfoService",
        "operation": "getJobOperInsttList",
    },

    "PUB-004_자립형일자리_사업모집공고": {
        "base": f"{XML_BASE}/JobBsnInfoService",
        "operation": "getJobBsnRecruitList",
    },

    "PUB-011_100세누리_구인정보": {
        "base": f"{XML_BASE}/SenuriService",
        "operation": "getJobList",
    },
}


# ============================================================
# 4. odcloud API 목록
# 연도별 엔드포인트 전체 등록
# ============================================================

OD_SOURCES = {

    "PUB-005_장기요양_인정현황": {
        "namespace": "3051420",
        "paths": {
            "2013Q4": "uddi:8bc5421b-b709-4e92-ab61-8a5e9571e4f0_201610261438",
            "2014Q1": "uddi:442633de-1cd7-46f1-90b7-9e3e98875938_201610261437",
            "2014Q2": "uddi:2e0aee8a-3c5b-4fd2-abb9-ca26005ef5c3_201610261439",
            "2020": "uddi:5795bcf6-479d-4ce9-94a6-bec812b41e04",
            "2021": "uddi:b48e683a-b31d-4750-afd7-ff8f04581180",
            "2022": "uddi:10da3de0-090f-47bc-ba60-f0ebe0ccc794",
            "2023": "uddi:a400c1d4-2358-4c37-8494-c169018bfd86",
            "2024": "uddi:76462a9a-5c61-4193-9b23-bd5ba862ade4",
            "2025": "uddi:14eb2dec-3ad4-4278-b0a6-fdf135dee3ec",
        },
    },

    "PUB-006_연령별_재해자수": {
        "namespace": "15064486",
        "paths": {
            "2020": "uddi:3d1647ed-bc3e-4c9c-8422-c36c72dbbd21",
            "2021": "uddi:f2304b91-8dbb-412e-bc26-911d27a5efb4",
            "2022": "uddi:d1484fb9-a10a-46e7-9bbf-f4743d8bf90e",
            "2023": "uddi:9f23f275-d0e6-484e-b353-e2f6c18c34b3",
            "2024": "uddi:ea441150-2e3a-4cf5-8649-9c9fd636d716",
            "2025": "uddi:8a87222e-14f0-4664-bfa6-f8c3f691a0f7",
        },
    },

    "PUB-007_산업중분류별_사고재해": {
        "namespace": "15084656",
        "paths": {
            "2020": "uddi:79613c8f-5eae-45d4-9f4f-d047edb084dd",
            "2021": "uddi:3e4d3a05-30a6-46bc-95d6-9e1ebf1d51df",
            "2022": "uddi:fbb7e7a5-9e47-4f90-8b3c-bb2bf2fbdd81",
            "2023": "uddi:c79506b4-7501-40f9-8f02-c12f2451ba22",
            "2024": "uddi:e0494ce1-7eba-4012-bed4-ed539c3a0914",
            "2025": "uddi:6051f73c-761d-4362-9b3a-a6eca6ac42a3",
        },
    },

    "PUB-008_노동지청별_재해자수": {
        "namespace": "15064475",
        "paths": {
            "2020": "uddi:45b708b3-a352-4c5c-bed7-db74df07129d",
            "2021": "uddi:b83119d1-f070-442c-b0b8-885cb9adc2ae",
            "2022": "uddi:f2c37b4f-65c2-476b-8559-e88430401155",
            "2023": "uddi:bd3cc240-a3ab-40a2-85cb-b3667f6856eb",
            "2024": "uddi:a6ae30d6-00bb-4fa1-acfc-fa19320e45a7",
            "2025": "uddi:c2dc05ff-1611-4395-b46a-b92cefc95070",
        },
    },

    "PUB-009_산업규모별_재해자수": {
        "namespace": "15064490",
        "paths": {
            "2020": "uddi:c99ee393-2d14-4b2f-99d3-a9ab4a0d63df",
            "2021": "uddi:395f1d49-c4e8-43c1-8234-a921e75661cf",
            "2022": "uddi:516495df-babb-400a-918f-9e9b046aee96",
            "2023": "uddi:1e446a30-618f-4e6e-8735-a7e5330f2275",
            "2024": "uddi:895c950e-dd0e-4968-ab1a-ee6183f59b0b",
            "2025": "uddi:5b822baf-2a25-4642-8bc9-d14a2aa458c9",
        },
    },

    "PUB-010_시도별_장기요양판정": {
        "namespace": "15072696",
        "paths": {
            "2015": "uddi:553b4d25-80c0-471b-9cc7-af7abee0b329",
            "2016": "uddi:b5b43d81-31d0-4b31-8c0f-9e83d6202da9",
            "2018": "uddi:ef2810b1-0856-4b7e-aa78-88e3c58c1a38",
            "2019": "uddi:b85af1c8-a290-42e1-b57f-ac91e5604e1a",
            "2020": "uddi:872f15c4-7f70-4c68-a8c9-8f22d2f79bcc",
            "2021": "uddi:63aa9ab2-863b-4a63-85cf-e9b43dccb240",
            "2022": "uddi:0131eccc-8fe1-4dd1-b83d-bc8824f6e939",
            "2024": "uddi:130307ee-0834-4fc9-b557-d74af9424e8b",
            "2025": "uddi:d8b661ea-f3eb-4717-8ede-afff5eca1e8a",
        },
    },
}


# ============================================================
# 5. 파일 저장 / 재사용
# ============================================================

def save_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    temp = path.with_suffix(".tmp")

    df.to_csv(
        temp,
        index=False,
        encoding="utf-8-sig",
    )

    temp.replace(path)


def read_saved(path):
    if path.exists() and not FORCE_REFRESH:
        try:
            df = pd.read_csv(
                path,
                dtype=str,
                keep_default_na=False,
                encoding="utf-8-sig",
            )
            if len(df) > 0:
                return df
        except Exception as e:
            print("저장 파일 읽기 실패:", path.name, e)

    return None


# ============================================================
# 6. 요청 함수
# ============================================================

def request_api(session, url, params):

    last_error = None

    for attempt in range(RETRIES + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=(
                    CONNECT_TIMEOUT,
                    READ_TIMEOUT,
                ),
            )

            response.raise_for_status()
            return response

        except requests.RequestException as e:
            last_error = e

            if attempt < RETRIES:
                print(
                    f"    요청 재시도 {attempt + 1}/{RETRIES}"
                )
                time.sleep(attempt + 1)

    raise last_error


# ============================================================
# 7. XML 응답 파싱
# ============================================================

def xml_to_rows(content):

    root = ET.fromstring(content)

    code = root.findtext(".//resultCode")

    if code and code not in ("00", "0", "0000"):
        message = root.findtext(".//resultMsg")
        raise ValueError(
            f"API 오류: {code} / {message}"
        )

    items = root.findall(".//item")

    rows = []

    for item in items:
        row = {}

        for child in item.iter():
            if child is item:
                continue

            if len(child) == 0:
                key = child.tag.split("}")[-1]
                row[key] = child.text

        if row:
            rows.append(row)

    total = root.findtext(".//totalCount")

    return rows, int(total) if total and total.isdigit() else None


# ============================================================
# 8. XML 수집
# ============================================================

def collect_xml(name, config):

    saved_path = RAW_DIR / f"{name}.csv"

    existing = read_saved(saved_path)

    if existing is not None:
        print(f"  [저장 파일 재사용] {len(existing):,}행")
        return existing, "CACHED"

    url = (
        config["base"].rstrip("/")
        + "/"
        + config["operation"]
    )

    rows = []
    signatures = set()
    completed = False

    with requests.Session() as session:

        for page in range(1, MAX_PAGES + 1):

            print(f"  페이지 {page} 요청", flush=True)

            response = request_api(
                session,
                url,
                {
                    "serviceKey": KEY,
                    "pageNo": page,
                    "numOfRows": PAGE_SIZE,
                },
            )

            batch, total = xml_to_rows(response.content)

            if not batch:
                completed = True
                break

            signature = hashlib.sha256(
                json.dumps(
                    batch,
                    ensure_ascii=False,
                    sort_keys=True,
                ).encode("utf-8")
            ).hexdigest()

            if signature in signatures:
                raise RuntimeError(
                    "동일한 페이지가 반복 반환되었습니다. "
                    "페이지 파라미터 또는 필수 검색조건을 확인하세요."
                )

            signatures.add(signature)

            rows.extend(batch)

            print(
                f"    누적 {len(rows):,}행",
                flush=True,
            )

            if total is not None and len(rows) >= total:
                completed = True
                break

            if total is None and len(batch) < PAGE_SIZE:
                completed = True
                break

    if not completed:
        raise RuntimeError(
            f"최대 {MAX_PAGES}페이지에 도달했습니다. "
            "데이터 유실 방지를 위해 완료 파일로 저장하지 않습니다."
        )

    if not rows:
        raise ValueError(
            "응답에서 데이터 행을 찾지 못했습니다. "
            "필수 요청변수 또는 XML 구조를 확인하세요."
        )

    df = pd.DataFrame(rows)

    df["source_dataset"] = name
    df["source_endpoint"] = url

    save_csv(df, saved_path)

    return df, "SUCCESS"


# ============================================================
# 9. odcloud 연도별 수집
# ============================================================

def collect_odcloud_period(
    name,
    namespace,
    period,
    endpoint,
):

    folder = RAW_DIR / name
    saved_path = folder / f"{period}.csv"

    existing = read_saved(saved_path)

    if existing is not None:
        print(
            f"  [{period}] 저장 파일 재사용 "
            f"{len(existing):,}행"
        )
        return existing, "CACHED"

    url = (
        f"{OD_BASE}/{namespace}/v1/{endpoint}"
    )

    rows = []
    signatures = set()
    completed = False

    with requests.Session() as session:

        for page in range(1, MAX_PAGES + 1):

            print(
                f"  [{period}] 페이지 {page}",
                flush=True,
            )

            response = request_api(
                session,
                url,
                {
                    "serviceKey": KEY,
                    "page": page,
                    "perPage": PAGE_SIZE,
                    "returnType": "JSON",
                },
            )

            body = response.json()

            batch = body.get("data", [])

            if not isinstance(batch, list):
                raise ValueError(
                    "JSON data 필드가 리스트가 아닙니다."
                )

            if not batch:
                completed = True
                break

            signature = hashlib.sha256(
                json.dumps(
                    batch,
                    ensure_ascii=False,
                    sort_keys=True,
                ).encode("utf-8")
            ).hexdigest()

            if signature in signatures:
                raise RuntimeError(
                    "동일한 페이지가 반복 반환되었습니다."
                )

            signatures.add(signature)

            rows.extend(batch)

            total = body.get(
                "matchCount",
                body.get("totalCount"),
            )

            if total is not None and len(rows) >= int(total):
                completed = True
                break

            if total is None and len(batch) < PAGE_SIZE:
                completed = True
                break

    if not completed:
        raise RuntimeError(
            f"{MAX_PAGES}페이지 제한에 도달했습니다."
        )

    if not rows:
        raise ValueError("조회된 데이터가 없습니다.")

    df = pd.DataFrame(rows)

    df["source_period"] = period
    df["source_endpoint"] = url
    df["source_dataset"] = name

    save_csv(df, saved_path)

    return df, "SUCCESS"


# ============================================================
# 10. 전체 수집 실행
# ============================================================

all_datasets = {}
collection_status = []
period_status = []

print("=" * 65)
print("IF_v2 공공데이터 전체 수집 시작")
print("=" * 65)

# ------------------------------------------------------------
# XML 6개 작업
# ------------------------------------------------------------

for name, config in XML_SOURCES.items():

    print(f"\n[XML] {name}", flush=True)

    try:
        df, status = collect_xml(name, config)

        all_datasets[name] = df

        collection_status.append({
            "dataset": name,
            "status": status,
            "rows": len(df),
            "columns": len(df.columns),
        })

        print(
            f"  완료: {len(df):,}행 / "
            f"{len(df.columns)}열"
        )

    except Exception as e:

        print(f"  실패: {type(e).__name__}: {e}")

        collection_status.append({
            "dataset": name,
            "status": f"FAILED: {type(e).__name__}: {e}",
            "rows": 0,
            "columns": 0,
        })


# ------------------------------------------------------------
# odcloud 6개 데이터셋
# ------------------------------------------------------------

for name, config in OD_SOURCES.items():

    print(f"\n[JSON] {name}", flush=True)

    frames = []
    failed_periods = []

    for period, endpoint in config["paths"].items():

        try:
            df, status = collect_odcloud_period(
                name,
                config["namespace"],
                period,
                endpoint,
            )

            frames.append(df)

            period_status.append({
                "dataset": name,
                "period": period,
                "status": status,
                "rows": len(df),
            })

            print(
                f"  [{period}] 완료: {len(df):,}행"
            )

        except Exception as e:

            failed_periods.append(period)

            print(
                f"  [{period}] 실패: "
                f"{type(e).__name__}: {e}"
            )

            period_status.append({
                "dataset": name,
                "period": period,
                "status": f"FAILED: {type(e).__name__}: {e}",
                "rows": 0,
            })

    if frames:

        merged = pd.concat(
            frames,
            ignore_index=True,
            sort=False,
        )

        all_datasets[name] = merged

        # 모든 연도가 성공했을 때만 완전한 통합본 저장
        if not failed_periods:
            save_csv(
                merged,
                MERGED_DIR / f"{name}.csv",
            )
            dataset_status = "SUCCESS"
        else:
            dataset_status = (
                "PARTIAL: "
                + ",".join(failed_periods)
            )

        collection_status.append({
            "dataset": name,
            "status": dataset_status,
            "rows": len(merged),
            "columns": len(merged.columns),
        })

        print(
            f"  통합: {len(merged):,}행 / "
            f"{len(merged.columns)}열"
        )

    else:

        collection_status.append({
            "dataset": name,
            "status": "FAILED",
            "rows": 0,
            "columns": 0,
        })


# ============================================================
# 11. XML 통합본도 저장
# ============================================================

for name in XML_SOURCES:

    if name in all_datasets:

        save_csv(
            all_datasets[name],
            MERGED_DIR / f"{name}.csv",
        )


# ============================================================
# 12. 컬럼 프로파일 생성
# ============================================================

column_rows = []

for name, df in all_datasets.items():

    clean = df.replace(
        r"^\s*$",
        pd.NA,
        regex=True,
    )

    for column in clean.columns:

        series = clean[column]

        column_rows.append({
            "dataset": name,
            "column": column,
            "dtype": str(series.dtype),
            "total_rows": len(series),
            "null_count": int(series.isna().sum()),
            "null_rate_pct": round(
                series.isna().mean() * 100,
                2,
            ),
            "unique_count": int(
                series.nunique(dropna=True)
            ),
            "sample_values": " | ".join(
                series.dropna()
                .astype(str)
                .unique()[:3]
            ),
        })

column_profile = pd.DataFrame(column_rows)


# ============================================================
# 13. 수집 결과 저장
# ============================================================

status_df = pd.DataFrame(collection_status)
period_df = pd.DataFrame(period_status)

save_csv(
    status_df,
    REPORT_DIR / "collection_status.csv",
)

save_csv(
    period_df,
    REPORT_DIR / "period_status.csv",
)

if not column_profile.empty:
    save_csv(
        column_profile,
        REPORT_DIR / "column_profile.csv",
    )


# ============================================================
# 14. 최종 결과
# ============================================================

print("\n" + "=" * 65)
print("전체 수집 결과")
print("=" * 65)

display(status_df)

print("\n연도별 수집 결과")
display(period_df)

print("\n확보한 데이터셋")
for name, df in all_datasets.items():
    print(
        f"{name}: "
        f"{len(df):,}행 × {len(df.columns)}열"
    )

print("\n총 수집 작업:", len(XML_SOURCES) + len(OD_SOURCES))
print("데이터셋 확보:", len(all_datasets))
print("완전 수집:", int(status_df["status"].isin(["SUCCESS", "CACHED"]).sum()))
print("저장 위치:", BASE_DIR)

# 기존 노트북 변수와 호환
job_datasets = {
    k: v for k, v in all_datasets.items()
    if k in XML_SOURCES
}

odcloud_datasets = {
    k: v for k, v in all_datasets.items()
    if k in OD_SOURCES
}

환경설정 파일 확인 완료
저장 경로: /Users/jeong-yelim/IF_portfolio/data/output/public_api
IF_v2 공공데이터 전체 수집 시작

[XML] PUB-001_자립형일자리_행정구역코드
  페이지 1 요청
    누적 19행
  완료: 19행 / 7열

[XML] PUB-002A_노인사회활동_지역코드
  페이지 1 요청
    누적 340행
  완료: 340행 / 6열

[XML] PUB-002B_노인사회활동_기관코드
  페이지 1 요청
    누적 1,000행
  페이지 2 요청
    누적 2,000행
  페이지 3 요청
    누적 3,000행
  페이지 4 요청
    누적 3,935행
  완료: 3,935행 / 4열

[XML] PUB-003_자립형일자리_수행기관
  페이지 1 요청
    누적 1,000행
  페이지 2 요청
    누적 1,485행
  완료: 1,485행 / 10열

[XML] PUB-004_자립형일자리_사업모집공고
  페이지 1 요청
    누적 1,000행
  페이지 2 요청
    누적 2,000행
  페이지 3 요청
    누적 3,000행
  페이지 4 요청
    누적 4,000행
  페이지 5 요청
    누적 5,000행
  페이지 6 요청
    누적 6,000행
  페이지 7 요청
    누적 7,000행
  페이지 8 요청
    누적 8,000행
  페이지 9 요청
    누적 9,000행
  페이지 10 요청
    누적 10,000행
  페이지 11 요청
    누적 11,000행
  페이지 12 요청
    누적 12,000행
  페이지 13 요청
    누적 13,000행
  페이지 14 요청
    누적 14,000행
  페이지 15 요청
    누적 15,000행
  페이지 16 요청
    누적 16,000행
  페이지 17 요청
    누적 17,000행
  페이지 18 요청
    누적 18,000행
  페이지 19 요청
    누적 19,000행
  페이

,dataset,status,rows,columns
0,PUB-001_자립형일자리_행정구역코드,SUCCESS,19,7
1,PUB-002A_노인사회활동_지역코드,SUCCESS,340,6
2,PUB-002B_노인사회활동_기관코드,SUCCESS,3935,4
3,PUB-003_자립형일자리_수행기관,SUCCESS,1485,10
4,PUB-004_자립형일자리_사업모집공고,FAILED: RuntimeError: 최대 30페이지에 도달했습니다. 데이터 유실...,0,0
5,PUB-011_100세누리_구인정보,FAILED: RuntimeError: 동일한 페이지가 반복 반환되었습니다. 페이지...,0,0
6,PUB-005_장기요양_인정현황,SUCCESS,210,19
7,PUB-006_연령별_재해자수,SUCCESS,82,44
8,PUB-007_산업중분류별_사고재해,SUCCESS,180,45
9,PUB-008_노동지청별_재해자수,SUCCESS,294,44



연도별 수집 결과


,dataset,period,status,rows
0,PUB-005_장기요양_인정현황,2013Q4,SUCCESS,10
1,PUB-005_장기요양_인정현황,2014Q1,SUCCESS,12
2,PUB-005_장기요양_인정현황,2014Q2,SUCCESS,12
3,PUB-005_장기요양_인정현황,2020,SUCCESS,26
4,PUB-005_장기요양_인정현황,2021,SUCCESS,26
5,PUB-005_장기요양_인정현황,2022,SUCCESS,28
6,PUB-005_장기요양_인정현황,2023,SUCCESS,30
7,PUB-005_장기요양_인정현황,2024,SUCCESS,32
8,PUB-005_장기요양_인정현황,2025,SUCCESS,34
9,PUB-006_연령별_재해자수,2020,SUCCESS,15



확보한 데이터셋
PUB-001_자립형일자리_행정구역코드: 19행 × 7열
PUB-002A_노인사회활동_지역코드: 340행 × 6열
PUB-002B_노인사회활동_기관코드: 3,935행 × 4열
PUB-003_자립형일자리_수행기관: 1,485행 × 10열
PUB-005_장기요양_인정현황: 210행 × 19열
PUB-006_연령별_재해자수: 82행 × 44열
PUB-007_산업중분류별_사고재해: 180행 × 45열
PUB-008_노동지청별_재해자수: 294행 × 44열
PUB-009_산업규모별_재해자수: 180행 × 36열
PUB-010_시도별_장기요양판정: 8,001행 × 8열

총 수집 작업: 12
데이터셋 확보: 10
완전 수집: 10
저장 위치: /Users/jeong-yelim/IF_portfolio/data/output/public_api
